# 11A Appendix: Multinomial Logistic Regression and Softmax


## From Two Classes to $K$

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/11A-Appendix-Multinomial-Logistic-Regression.ipynb)

> Draft -- Under review

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| code-fold: true
#| code-summary: "Imports and setup"
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from slideUtilities import OKABE_ITO, class_cmaps

warnings.filterwarnings('ignore')
%matplotlib inline
SEED = 701

[Lecture 11A](11A-Logistic-Regression.qmd) modelled a **binary** outcome: one
probability $p = \sigma(\boldsymbol\beta^T\mathbf{x})$, the sigmoid of one linear score.

* Many outcomes have **more than two** classes: a species, a product category, a digit, a
  wine cultivar.
* We want a model that returns **$K$ probabilities that sum to 1**, fitted the same way
  (maximum likelihood) and read the same way (log-odds).
* The answer is **multinomial logistic regression**: one linear score per class, turned into
  probabilities by the **softmax**. The sigmoid will turn out to be its $K=2$ special case.


This appendix extends Lecture 11A; it reuses that lecture's notation ($\boldsymbol\beta$,
$\mathbf{x}_i$, $p_i$, the log-likelihood $\ell$, the design matrix $X$) and assumes its
derivation of the binary gradient. It is also the bridge to
[Lecture 16](16-Neural-Networks.qmd): a neural-network classifier ends in exactly the softmax and
cross-entropy loss developed here. **Quiz scope:** you should be able to say that the softmax
turns $K$ scores into probabilities that sum to 1 and that with $K=2$ it reduces to the sigmoid.
The rest is background, not quiz material.


## The Example: Three Wine Cultivars

The UCI wine data: 178 wines from **three cultivars** grown in the same region of Italy, with 13
chemical measurements. We use two of them, so every picture is a plane.

In [ ]:
#| code-fold: true
wine = load_wine(as_frame=True)
features = ['alcohol', 'flavanoids']
df = wine.frame[features + ['target']].rename(columns={'target': 'cultivar'})
K = df['cultivar'].nunique()
print(df.groupby('cultivar')[features].mean().round(2).assign(n=df['cultivar'].value_counts().sort_index()))

In [ ]:
#| code-fold: true
#| fig-align: center
bold, light = class_cmaps(K)
fig, ax = plt.subplots(figsize=(5.5, 2.7))
for k in range(K):
    sub = df[df['cultivar'] == k]
    ax.scatter(sub['alcohol'], sub['flavanoids'], color=OKABE_ITO[k], edgecolor='k',
               linewidth=0.4, s=30, label=f'cultivar {k}')
ax.set_xlabel('alcohol (%)')
ax.set_ylabel('flavanoids')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## First Idea: One-vs-Rest

Reuse the binary model: fit **$K$ separate** logistic regressions, "cultivar $k$ or not?", and
predict the class whose model is most confident.

In [ ]:
#| code-fold: true
X = df[features].to_numpy()
y = df['cultivar'].to_numpy()
ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(X, y)
raw = np.column_stack([est.predict_proba(X)[:, 1] for est in ovr.estimators_])
row_sums = raw.sum(axis=1)
print(f"sum of the {K} binary probabilities per wine: "
      f"min {row_sums.min():.2f}, median {np.median(row_sums):.2f}, max {row_sums.max():.2f}")

* It works as a *classifier*, but the $K$ probabilities **do not sum to 1**: each model was
  fitted on its own and knows nothing about the others.
* scikit-learn's `OneVsRestClassifier.predict_proba` hides this by dividing each row by its sum
  afterwards, a patch rather than a model.
* We want **one** model whose probabilities are coherent by construction.


## Log-Odds Against a Reference Class

In 11A the linear score was the log-odds of class 1 against class 0. With $K$ classes, pick one
**reference class** (say class $0$) and give every other class its own coefficient vector:

$$
\log\frac{P(y = k \mid \mathbf{x})}{P(y = 0 \mid \mathbf{x})} = \boldsymbol\beta_k^T\mathbf{x},
\qquad k = 1, \dots, K-1.
$$

Write $p_k$ for $P(y = k \mid \mathbf{x})$. Exponentiating gives $p_k = p_0\, e^{\boldsymbol\beta_k^T\mathbf{x}}$,
and the probabilities must sum to 1, so $p_0\big(1 + \sum_{j=1}^{K-1} e^{\boldsymbol\beta_j^T\mathbf{x}}\big) = 1$:

$$
p_0 = \frac{1}{1 + \sum_{j=1}^{K-1} e^{\boldsymbol\beta_j^T\mathbf{x}}},
\qquad
p_k = \frac{e^{\boldsymbol\beta_k^T\mathbf{x}}}{1 + \sum_{j=1}^{K-1} e^{\boldsymbol\beta_j^T\mathbf{x}}}.
$$


This is exactly how 11A built the binary model, run $K-1$ times: a linear model for each
log-odds, inverted to probabilities. With $K = 2$ there is one log-odds, $\boldsymbol\beta_1^T\mathbf{x}$,
and $p_1 = e^{z}/(1+e^{z}) = \sigma(z)$, the sigmoid.


## The Softmax

Set $\boldsymbol\beta_0 = \mathbf{0}$, so $e^{\boldsymbol\beta_0^T\mathbf{x}} = 1$, and the
formula becomes symmetric in the classes. With one **score** $z_k = \boldsymbol\beta_k^T\mathbf{x}$
per class:

$$
p_k = \operatorname{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}}.
$$

* Exponentiate every score (now all positive), then divide by the total (now they sum to 1).
* It preserves order: the largest score gets the largest probability. "Soft" because the
  runner-up still gets some: it is a smooth version of `argmax`.
* Only **differences** of scores matter: adding the same constant to every $z_k$ changes nothing.


## The Softmax in NumPy

In [ ]:
#| code-fold: false
def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))   # subtracting the max changes nothing, avoids overflow
    return e / e.sum(axis=-1, keepdims=True)

print(softmax([2.0, 1.0, 0.1]).round(3))
print(softmax([102.0, 101.0, 100.1]).round(3))      # same differences, same probabilities

Computing $e^{1000}$ overflows, but the softmax of $(1000, 999)$ is perfectly ordinary. Because
only differences matter, subtracting the largest score first gives the same answer with every
exponent $\le 0$. Library implementations do the same.


## Softmax with $K = 2$ Is the Sigmoid

With two scores,

$$
\operatorname{softmax}(z_0, z_1)_1 = \frac{e^{z_1}}{e^{z_0} + e^{z_1}} = \frac{1}{1 + e^{-(z_1 - z_0)}} = \sigma(z_1 - z_0).
$$

* Only the difference $z_1 - z_0 = (\boldsymbol\beta_1 - \boldsymbol\beta_0)^T\mathbf{x}$ matters,
  so two coefficient vectors carry the information of one.
* Fixing $\boldsymbol\beta_0 = \mathbf{0}$ gives back 11A's model, $p = \sigma(\boldsymbol\beta^T\mathbf{x})$.
* **Binary logistic regression is multinomial logistic regression with $K = 2$.**


In [ ]:
#| code-fold: false
z = np.linspace(-6, 6, 7)
two_class = softmax(np.column_stack([np.zeros_like(z), z]))[:, 1]
print(np.allclose(two_class, 1 / (1 + np.exp(-z))))

## Too Many Parameters: Pick a Reference

The symmetric form has $K$ coefficient vectors, but adding the same vector $\mathbf{c}$ to every
$\boldsymbol\beta_k$ adds $\mathbf{c}^T\mathbf{x}$ to every score and leaves every probability
unchanged.

* The $K$ vectors are **not identifiable**: infinitely many settings give the same model. Only
  $K-1$ vectors' worth of information is real.
* **statsmodels** (`MNLogit`) fixes it by setting the first class's coefficients to zero: every
  reported coefficient is *relative to class 0*.
* **scikit-learn** keeps all $K$ vectors and lets its default L2 penalty choose among them: the
  penalty picks the setting with the smallest $\sum_k\|\boldsymbol\beta_k\|^2$, which makes
  the coefficients sum to zero across classes.


This is why the coefficients from the two libraries look different even when the predicted
probabilities agree: they are two parameterizations of the same set of models. Differences
between classes, $\boldsymbol\beta_k - \boldsymbol\beta_j$, are the same in both (up to the effect
of scikit-learn's penalty), and those are what the probabilities depend on.


## The Likelihood: One-Hot Labels

Encode each label as a **one-hot** vector: $y_{ik} = 1$ if observation $i$ is in class $k$, else $0$.
The probability of what was observed is $\prod_k p_{ik}^{\,y_{ik}}$, which picks out the
probability of the true class. Over independent observations:

$$
\ell(\boldsymbol\beta_0, \dots, \boldsymbol\beta_{K-1}) = \sum_{i=1}^n \sum_{k=0}^{K-1} y_{ik}\log p_{ik},
\qquad p_{ik} = \operatorname{softmax}(\mathbf{z}_i)_k,\ \ z_{ik} = \boldsymbol\beta_k^T\mathbf{x}_i.
$$

* $-\ell/n$ is the **multiclass log-loss**, or **cross-entropy**: the average of
  $-\log(\text{probability given to the true class})$.
* With $K=2$ the inner sum is $y_i\log p_i + (1-y_i)\log(1-p_i)$: 11A's log-likelihood.
* It is concave in the $\boldsymbol\beta_k$, so there is still one best fit (once the reference is fixed), and no closed form.


This is the loss in [Lecture 16](16-Neural-Networks.qmd)'s "Classification" panel,
$L = -\frac{1}{N}\sum_i\sum_c y_{ic}\log\hat y_{ic}$, letter for letter.


## The Gradient Has the Same Shape

For each class $k$:

$$
\nabla_{\boldsymbol\beta_k}\,\ell = \sum_{i=1}^n \big(y_{ik} - p_{ik}\big)\,\mathbf{x}_i .
$$

Stack the coefficient vectors as the columns of a $(d+1) \times K$ matrix $B$, the one-hot labels
as an $n \times K$ matrix $Y$, and the probabilities as $P = \operatorname{softmax}(XB)$ (row by row):

$$
\nabla_B\,\ell = X^T(Y - P).
$$


Compare 11A: $\nabla_{\boldsymbol\beta}\,\ell = X^T(\mathbf{y} - \mathbf{p})$. Each class's
coefficients move in the direction of the inputs, weighted by **how much the observed indicator
exceeds the predicted probability**.



## Deriving the gradient

The derivation follows 11A's five steps, with the sigmoid's derivative replaced by the softmax's.

### Step 1: The derivative of the softmax

For $p_k = e^{z_k}/S$ with $S = \sum_j e^{z_j}$, the quotient rule gives

$$
\frac{\partial p_k}{\partial z_j} =
\begin{cases}
\dfrac{e^{z_k}S - e^{z_k}e^{z_k}}{S^2} = p_k(1 - p_k), & j = k,\\[2ex]
\dfrac{0 - e^{z_k}e^{z_j}}{S^2} = -p_k p_j, & j \ne k,
\end{cases}
\qquad\text{i.e.}\qquad
\frac{\partial p_k}{\partial z_j} = p_k(\delta_{kj} - p_j),
$$

where $\delta_{kj}$ is 1 if $j = k$ and 0 otherwise. With $K = 2$ the diagonal case is 11A's
$\sigma' = \sigma(1-\sigma)$.

### Step 2: The derivative of one observation's log-likelihood with respect to a score

Observation $i$ contributes $\ell_i = \sum_k y_{ik}\log p_{ik}$. By the chain rule,

$$
\frac{\partial \ell_i}{\partial z_{ij}} = \sum_k \frac{y_{ik}}{p_{ik}}\, p_{ik}(\delta_{kj} - p_{ij})
= \sum_k y_{ik}\delta_{kj} - p_{ij}\sum_k y_{ik} = y_{ij} - p_{ij},
$$

using $\sum_k y_{ik} = 1$ (exactly one class is observed).

### Step 3: From scores to coefficients

The score $z_{ij} = \boldsymbol\beta_j^T\mathbf{x}_i$ depends on $\boldsymbol\beta_j$ only, with
$\nabla_{\boldsymbol\beta_j} z_{ij} = \mathbf{x}_i$, so

$$
\nabla_{\boldsymbol\beta_j}\,\ell_i = (y_{ij} - p_{ij})\,\mathbf{x}_i .
$$

### Step 4: Sum over observations

$$
\nabla_{\boldsymbol\beta_j}\,\ell = \sum_{i=1}^n (y_{ij} - p_{ij})\,\mathbf{x}_i
= X^T(\mathbf{y}_{\cdot j} - \mathbf{p}_{\cdot j}),
$$

the $j$-th column of $X^T(Y - P)$. The softmax derivative was messier than the sigmoid's, but
the cancellation in Step 2 leaves exactly the same "observed minus predicted" form.


## Fitting It: Gradient Ascent by Hand

Standardize the two features, add the column of 1s, and climb $\ell$ exactly as in 11A, now
with a matrix of coefficients:

In [ ]:
#| code-fold: false
Xs = StandardScaler().fit_transform(X)
Xd = np.column_stack([np.ones(len(Xs)), Xs])          # n x (d+1) design matrix
Y = np.eye(K)[y]                                      # n x K one-hot labels

B = np.zeros((Xd.shape[1], K))                        # (d+1) x K, one column per class
eta = 0.5
for step in range(20_000):
    P = softmax(Xd @ B)
    B += eta * Xd.T @ (Y - P) / len(Xd)               # gradient ascent on l / n
B -= B[:, [0]]                                        # express relative to class 0

log_loss = -np.mean(np.log(softmax(Xd @ B)[np.arange(len(y)), y]))
print(f"log-loss {log_loss:.4f}")
print(pd.DataFrame(B, index=['intercept'] + features,
                   columns=[f'cultivar {k}' for k in range(K)]).round(3))

The update is 11A's $\boldsymbol\beta \leftarrow \boldsymbol\beta + \eta\, X^T(\mathbf{y}-\mathbf{p})$
with matrices in place of vectors. Without a reference class the climb would still converge in
probabilities, but $B$ could drift along the unidentifiable direction; subtracting column 0 at the
end puts the answer in statsmodels' parameterization so the two can be compared.


## The Same Fit in statsmodels

`MNLogit` maximizes the same $\ell$ with class 0 as the reference:

In [ ]:
#| code-fold: false
mnl = sm.MNLogit(y, sm.add_constant(Xs)).fit(disp=False)
print(np.abs(mnl.params - B[:, 1:]).max().round(4), "= largest difference from the hand-rolled fit")
print(f"log-loss {-mnl.llf / len(y):.4f}")

Same coefficients, same log-loss: there is nothing in `MNLogit` beyond the softmax, the
log-likelihood and a better optimizer. Plain gradient steps needed 20,000 iterations to agree to
four decimals; Newton's method, which `MNLogit` uses, needs a handful.


## Reading the Coefficients

Refit on the **original units** so a coefficient is "per 1% alcohol" or "per unit of flavanoids":

In [ ]:
#| code-fold: true
mnl_raw = sm.MNLogit(df['cultivar'], sm.add_constant(df[features])).fit(disp=False)
coef = mnl_raw.params.copy()
coef.columns = [f'cultivar {k} vs 0' for k in range(1, K)]
print(coef.round(2))
rrr = pd.concat({'per 1 unit': np.exp(coef.loc[features]),
                 'per 0.1 unit': np.exp(0.1 * coef.loc[features])}, axis=1)
print("\nexp(coefficient), relative-risk ratios:")
print(rrr.map(lambda v: f"{v:.3g}"))

* $\beta_{kj}$ is the change in the **log-odds of class $k$ against the reference class** for a
  one-unit increase in $x_j$, other inputs held fixed: 11A's reading, with "against class 0" added.
* $e^{\beta_{kj}}$ multiplies the ratio $p_k/p_0$ (statsmodels calls it a *relative-risk ratio*).
* Comparing two non-reference classes: subtract, $\beta_{kj} - \beta_{mj}$.
* **Change the reference class and every number in the table changes**, while the fitted
  probabilities do not. Always say what the reference is.


The flavanoids coefficient for cultivar 2 against cultivar 0 is large because flavanoids nearly
separate those two cultivars (mean 0.78 against 2.98), so one full unit is an enormous step on
that comparison and its ratio rounds to almost zero. Reading the ratio for a step the data
actually spans, 0.1 units, is more informative. Coefficients that keep growing as the optimizer
runs are the warning sign of *perfect* separation, where the maximum-likelihood estimate does
not exist; the L2 penalty of [Lecture 11B](11B-Regularization.qmd) is the usual fix.


## Decision Regions

Classify by the most probable class: $\hat y = \arg\max_k p_k = \arg\max_k z_k$ (the softmax keeps the order).

In [ ]:
#| code-fold: true
#| fig-align: center
clf = LogisticRegression(max_iter=1000).fit(X, y)    # multinomial by default
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 400),
                     np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 400))
grid = np.column_stack([xx.ravel(), yy.ravel()])
proba = clf.predict_proba(grid)
pred = proba.argmax(axis=1).reshape(xx.shape)
conf = proba.max(axis=1).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].contourf(xx, yy, pred, levels=np.arange(K + 1) - 0.5, cmap=light)
axes[0].set_title('argmax regions: boundaries are straight lines')
cs = axes[1].contourf(xx, yy, conf, levels=np.linspace(1 / K, 1, 8), cmap='Greys')
fig.colorbar(cs, ax=axes[1], label='largest probability')
axes[1].set_title('confidence: lowest near the boundaries')
for ax in axes:
    for k in range(K):
        m = y == k
        ax.scatter(X[m, 0], X[m, 1], color=OKABE_ITO[k], edgecolor='k', linewidth=0.4, s=22,
                   label=f'cultivar {k}')
    ax.set_xlabel('alcohol (%)')
axes[0].set_ylabel('flavanoids')
axes[0].legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

Classes $k$ and $j$ tie where $z_k = z_j$, that is where $(\boldsymbol\beta_k - \boldsymbol\beta_j)^T\mathbf{x} = 0$:
a **hyperplane**, as in 11A. Each class's region is the set of points where its score beats
every other, an intersection of half-spaces, so the regions are **convex** with straight edges.
As in the binary case, the nonlinearity lives in the confidence (right panel), not in the boundaries.


## In scikit-learn

In [ ]:
#| code-fold: false
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED)

multi = LogisticRegression(max_iter=1000).fit(X_train, y_train)
one_vs_rest = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(X_train, y_train)

for name, model in [('multinomial', multi), ('one-vs-rest', one_vs_rest)]:
    p = model.predict_proba(X_test)
    print(f"{name:12s} test log-loss {metrics.log_loss(y_test, p):.3f}   "
          f"accuracy {metrics.accuracy_score(y_test, p.argmax(axis=1)):.3f}")
print("rows of predict_proba sum to 1:", np.allclose(multi.predict_proba(X_test).sum(axis=1), 1))

* The multinomial fit has the lower log-loss: its probabilities are better. The accuracies
  differ by one wine out of 54, which is noise.
* `LogisticRegression` fits the **multinomial** model automatically when `y` has more than two
  classes. There is no `multi_class` argument in current scikit-learn; for one-vs-rest, wrap it
  in `OneVsRestClassifier`.
* `coef_` has shape $K \times d$: one row per class, the symmetric parameterization.
* Its default `C=1.0` is an L2 penalty (Lecture 11B), which also settles the identifiability problem.


## Evaluating $K$ Classes

In [ ]:
#| code-fold: true
y_pred = multi.predict(X_test)
majority = np.bincount(y_train).argmax()
print(f"majority-class baseline accuracy: {np.mean(y_test == majority):.3f}\n")
print(pd.DataFrame(metrics.confusion_matrix(y_test, y_pred),
                   index=[f'true {k}' for k in range(K)],
                   columns=[f'pred {k}' for k in range(K)]))
print()
print(metrics.classification_report(y_test, y_pred, digits=3))

* The confusion matrix is $K \times K$: rows are true classes, the diagonal is correct.
* Precision and recall are now **per class** (each class against the rest).
* Two ways to average them: **macro** (every class counts equally) and **weighted** (by class
  size). With imbalanced classes they can disagree a lot; macro exposes a neglected small class.
* Accuracy's floor is now the **largest** class's share, not one of two.


## Thresholds Become a Cost Matrix

With two classes, 11A's rule was: say "yes" when $p > C_{FP}/(C_{FP}+C_{FN})$. With $K$ classes
there is no single threshold. Let $C_{kj}$ be the cost of predicting $k$ when the truth is $j$, and
predict the class with the **lowest expected cost**:

$$
\hat y = \arg\min_k \sum_{j} C_{kj}\, p_j .
$$

In [ ]:
#| code-fold: true
C = np.ones((K, K)) - np.eye(K)          # every mistake costs 1 ...
C[1:, 0] = 5                             # ... except calling a cultivar-0 wine something else
proba_test = multi.predict_proba(X_test)
y_cost = (proba_test @ C.T).argmin(axis=1)
print(pd.DataFrame(C, index=[f'predict {k}' for k in range(K)],
                   columns=[f'true {j}' for j in range(K)]))
print(f"\npredictions changed by the cost matrix: {np.sum(y_cost != y_pred)} of {len(y_test)}")
print(f"cultivar-0 recall: argmax {metrics.recall_score(y_test, y_pred, labels=[0], average=None)[0]:.3f}, "
      f"cost-aware {metrics.recall_score(y_test, y_cost, labels=[0], average=None)[0]:.3f}")

With all off-diagonal costs equal, minimizing expected cost is exactly `argmax`. With $K=2$ and
costs $C_{10} = C_{FP}$, $C_{01} = C_{FN}$, the rule reduces to 11A's threshold $t^*$. The point
carries over unchanged: the probabilities come from the model, the costs come from the problem,
and the decision needs both.


## Where This Shows Up Next

* **Neural networks** ([Lecture 16](16-Neural-Networks.qmd)): a classifier's last layer computes
  one score per class and applies the softmax; training minimizes this cross-entropy. A network is
  multinomial logistic regression on features the hidden layers *learn*, rather than on the raw inputs.
* **Text** ([Lecture 18](18-NLP.qmd)): multinomial logistic regression on TF-IDF features is a
  strong baseline for multi-class document classification.
* **Ordered classes** (1–5 stars, small/medium/large): the multinomial model ignores the order.
  *Ordinal* logistic regression uses it; it is not covered in this course.


## Summary

1. **Model:** one linear score per class, $z_k = \boldsymbol\beta_k^T\mathbf{x}$, turned into
   probabilities by the softmax $p_k = e^{z_k}/\sum_j e^{z_j}$. It comes from modelling each
   class's log-odds against a reference class.
2. **$K = 2$ gives the sigmoid**, so 11A's model is the special case.
3. **Identifiability:** only differences of scores matter; statsmodels fixes a reference class,
   scikit-learn lets the L2 penalty choose. Coefficients are log-odds against the reference.
4. **Fit:** maximize $\ell = \sum_i\sum_k y_{ik}\log p_{ik}$ (minus the cross-entropy) with gradient
   $X^T(Y - P)$, the same "observed minus predicted" form as the binary case.
5. **Use:** boundaries are still hyperplanes; evaluate with a $K\times K$ confusion matrix and
   per-class or macro metrics; decide with a cost matrix, not a threshold.
